# NeuralMath — rede neural do zero

Notebook complementar do **NeuralMath.org** e do texto pedagógico *A anatomia matemática de uma rede neural*. Este notebook reproduz o exemplo atual do artigo: 2 entradas (massa e diâmetro), **12 neurônios ReLU na camada oculta**, uma saída sigmoide, **64 exemplos sintéticos de treinamento**, 8 exemplos ilustrativos de teste, erro quadrático, retropropagação explícita e descida do gradiente estocástica.

O código é deliberadamente explícito, com nomes em português, para que cada operação matemática possa ser inspecionada e relacionada às equações.

## 1. Ingredientes do modelo

Para um vetor de entrada padronizado $\mathbf{x}$, a rede calcula

$$
\mathbf{z}^{(1)}=W^{(1)}\mathbf{x}+\mathbf{b}^{(1)},\qquad
\mathbf{h}=\operatorname{ReLU}(\mathbf{z}^{(1)}),
$$

$$
z^{(2)}=W^{(2)}\mathbf{h}+b^{(2)},\qquad
\widehat y=\sigma(z^{(2)}).
$$

Implementamos diretamente as funções de ativação e suas derivadas; não usamos diferenciação automática.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def relu(s):
    return np.maximum(0, s)


def relu_derivada(s):
    return (s > 0).astype(float)


def sigmoide(s):
    return 1 / (1 + np.exp(-s))


def fronteira_base(massa):
    return (
        6.68
        + 0.74 * np.exp(-((massa - 155) / 20) ** 2)
        + 1.50 / (1 + np.exp(-(massa - 202) / 5))
    )


## 2. Dados sintéticos e padronização

Os dados são didáticos, mas permanecem em faixas plausíveis de massa e diâmetro. A rede nunca recebe a curva de referência usada para construir a geometria. Médias e desvios-padrão são calculados somente com o conjunto de treinamento.

In [ ]:
def construir_dados():
    massas_1 = np.linspace(118, 212, 16)
    massas_2 = massas_1 + np.array([
        1.2, -1.1, 0.8, -1.3, 1.0, -0.8, 1.1, -1.0,
        1.3, -0.9, 0.9, -1.2, 1.0, -1.0, 0.8, -0.7,
    ])
    k = np.arange(16)

    macas_1 = np.column_stack([
        massas_1,
        fronteira_base(massas_1) + 0.22 + 0.05 * np.sin(0.8 * k),
    ])
    macas_2 = np.column_stack([
        massas_2,
        fronteira_base(massas_2) + 0.58 + 0.06 * np.cos(0.45 * k),
    ])
    laranjas_1 = np.column_stack([
        massas_1,
        fronteira_base(massas_1) - (0.22 + 0.04 * np.cos(0.7 * k)),
    ])
    laranjas_2 = np.column_stack([
        massas_2,
        fronteira_base(massas_2) - (0.55 + 0.05 * np.sin(0.5 * k)),
    ])

    X_treino = np.vstack([macas_1, macas_2, laranjas_1, laranjas_2])
    y_treino = np.array([1.0] * 32 + [0.0] * 32)

    X_teste = np.array([
        [121.0, 7.20], [135.0, 7.45], [152.0, 7.75], [171.0, 7.32],
        [121.0, 6.28], [152.0, 6.86], [189.0, 6.48], [208.0, 7.55],
    ])
    y_teste = np.array([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0])
    return X_treino, y_treino, X_teste, y_teste


X_treino, y_treino, X_teste, y_teste = construir_dados()
media = X_treino.mean(axis=0)
desvio = X_treino.std(axis=0)
Xn = (X_treino - media) / desvio

print('Conjunto de treinamento' + ':', X_treino.shape)
print('Conjunto de teste' + ':', X_teste.shape)
print('Média' + ':', media)
print('Desvio-padrão' + ':', desvio)


## 3. Treinamento com retropropagação explícita

Para o erro quadrático $E=\tfrac12(y-\widehat y)^2$, cada derivada é escrita explicitamente. Os parâmetros são atualizados após cada exemplo de treinamento, em ordem embaralhada.

In [ ]:
gerador = np.random.default_rng(42)
W1 = 0.1 * gerador.standard_normal((12, 2))
b1 = np.zeros(12)
W2 = 0.1 * gerador.standard_normal(12)
b2 = 0.0

taxa_aprendizado = 0.05
n_epocas = 6000
historico = []

for epoca in range(n_epocas):
    for i in gerador.permutation(len(Xn)):
        x = Xn[i]
        alvo = y_treino[i]

        # Propagação direta
        z1 = W1 @ x + b1
        h = relu(z1)
        z2 = W2 @ h + b2
        y_hat = sigmoide(z2)

        # Retropropagação
        delta2 = (y_hat - alvo) * y_hat * (1 - y_hat)
        dW2 = delta2 * h
        db2 = delta2

        delta1 = (delta2 * W2) * relu_derivada(z1)
        dW1 = np.outer(delta1, x)
        db1 = delta1

        # Atualização por descida do gradiente estocástica
        W1 -= taxa_aprendizado * dW1
        b1 -= taxa_aprendizado * db1
        W2 -= taxa_aprendizado * dW2
        b2 -= taxa_aprendizado * db2

    erros = []
    for x, alvo in zip(Xn, y_treino):
        h = relu(W1 @ x + b1)
        y_hat = sigmoide(W2 @ h + b2)
        erros.append(0.5 * (alvo - y_hat) ** 2)
    historico.append(float(np.mean(erros)))

print(f"Perda média final: {historico[-1]:.10f}")


## 4. Previsões e fronteira de decisão aprendida

Após o treinamento, $\widehat y\ge 0.5$ é classificado como maçã e $\widehat y<0.5$ como laranja. O nível $\widehat y=0.5$ é a fronteira de decisão aprendida.

In [ ]:
def prever(X):
    X_padronizado = (X - media) / desvio
    saidas = []
    for x in X_padronizado:
        h = relu(W1 @ x + b1)
        saidas.append(sigmoide(W2 @ h + b2))
    return np.array(saidas)


previsoes_treino = prever(X_treino)
previsoes_teste = prever(X_teste)

print(f"Acurácia no treinamento: {np.mean((previsoes_treino >= 0.5) == y_treino):.4f}")
print(f"Acurácia no teste ilustrativo: {np.mean((previsoes_teste >= 0.5) == y_teste):.4f}")
print('Saídas no teste' + ':', np.round(previsoes_teste, 6))

massas = np.linspace(116, 214, 500)
diametros = np.linspace(6.0, 8.85, 500)
M, D = np.meshgrid(massas, diametros)
grade = np.column_stack([M.ravel(), D.ravel()])
P = prever(grade).reshape(M.shape)

fig, ax = plt.subplots(figsize=(9.2, 6.3))
ax.contourf(
    M, D, P,
    levels=[0.0, 0.25, 0.50, 0.75, 1.0],
    colors=['#f3e0cb', '#ecd5c4', '#dbe7cf', '#cde3bf'],
    alpha=0.95,
)
ax.contour(M, D, P, levels=[0.5], colors=['#5a1a8a'], linewidths=2.2)

mascara = y_treino == 1.0
ax.scatter(X_treino[mascara, 0], X_treino[mascara, 1], marker='o', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='Maçã — treino')
mascara = y_treino == 0.0
ax.scatter(X_treino[mascara, 0], X_treino[mascara, 1], marker='s', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='Laranja — treino')
mascara = y_teste == 1.0
ax.scatter(X_teste[mascara, 0], X_teste[mascara, 1], marker='^', s=92,
           color='#1f77b4', label='Maçã — teste')
mascara = y_teste == 0.0
ax.scatter(X_teste[mascara, 0], X_teste[mascara, 1], marker='x', s=110,
           color='#ff7f0e', linewidths=2.0, label='Laranja — teste')

ax.set_xlim(116, 214)
ax.set_ylim(6.0, 8.85)
ax.set_xlabel('Massa (g)')
ax.set_ylabel('Diâmetro (cm)')
ax.set_title('Saída da rede neural após o treinamento', fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True)
fig.tight_layout()
plt.show()


## 5. O que observar

O gráfico final conecta a álgebra à geometria. As regiões sombreadas mostram as classes previstas pela rede e a curva destacada representa o nível aprendido $\widehat y=0.5$. Como os dados são sintéticos e construídos para fins didáticos, a acurácia apresentada ilustra o funcionamento do algoritmo e não constitui uma avaliação estatística de desempenho.